# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and must be accessed using entity `@id` fields.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their `@id`.

In [ ]:
# List available record sets and their fields
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s).\n")
for rs in record_sets:
    print(f"Record Set @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    print(f"  Fields:")
    for f in rs.fields:
        print(f"    - Field @id: {f.id}, Name: {f.name}, Data type: {f.data_type}")
    print()

## 2.1 Example: Preview Records
Preview the first few records of a record set. All entities are referenced by their `@id`.

In [ ]:
# Example: Print a sample of records from the first record set using its @id
if len(record_sets) > 0:
    rs = record_sets[0]
    print(f"Sample records for Record Set @id: {rs.id}")
    for i, record in enumerate(dataset.records(record_set=rs.id)):
        print(record)
        if i >= 2:
            break
else:
    print("No record sets found.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We use the `@id` of each record set and field whenever performing extraction and referencing columns.

In [ ]:
# Extract data from each record set and load into DataFrames keyed by their record set @id
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for record set {record_set_id}: {df.shape[0]} rows, {df.shape[1]} columns")

# Show available columns for the first record set
if record_set_ids:
    example_id = record_set_ids[0]
    print(f"\nColumns for record set {example_id}:")
    print(dataframes[example_id].columns.tolist())
    # Show the first rows
    display(dataframes[example_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing: Filter records based on specific criteria, normalize numeric fields, group and summarize data. All columns (fields) are referenced by their `@id`.

The code block below demonstrates selection, normalization, and grouping for a numeric field, using the field's `@id`.

In [ ]:
import numpy as np

# Example: Use the first record set and attempt to select a numeric field
if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    # Find numeric fields by their declared Croissant field data types
    rs = [rs for rs in record_sets if rs.id == record_set_id][0]
    numeric_field_id = None
    numeric_type_keywords = ["Integer", "Float", "Number"]  # Types from schema
    for field in rs.fields:
        if any(k.lower() in str(field.data_type).lower() for k in numeric_type_keywords):
            if field.id in df.columns:
                numeric_field_id = field.id
                break
    if numeric_field_id:
        print(f"Using numeric field with @id: {numeric_field_id}")
        # Drop NaNs and coerce values to numeric
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        # Threshold for filtering
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}: {len(filtered_df)} records of {len(df)}")
        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Try grouping by a categorical field
        group_field_id = None
        # Pick the first non-numeric, non-unique field (with <25 unique values perhaps)
        for field in rs.fields:
            if field.id != numeric_field_id and field.id in df.columns:
                unique_vals = df[field.id].nunique()
                if unique_vals < 25 and unique_vals > 1:
                    group_field_id = field.id
                    break
        if group_field_id:
            print(f"\nGrouping by field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(grouped_df.head())
        else:
            print("No suitable categorical group field found.")
    else:
        print("No numeric field found in the record set.")
else:
    print("No record set available.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we plot the distribution of the numeric field, and (if exists) show the mean value grouped by the chosen categorical field. All fields are referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30)
    plt.title(f'Distribution of numeric field (@id: {numeric_field_id})')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(8,5))
        sns.barplot(x=grouped_df.index, y=grouped_df[numeric_field_id].values)
        plt.title(f'Mean {numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.show()


## 6. Conclusion
In this notebook, we have loaded the dataset defined by the Croissant schema using the `mlcroissant` library, explored its structure (record sets, fields, and `@id`s), loaded data via those `@id` references, and performed simple EDA and visualization steps.

Key findings and approaches:
- All references to data elements (record sets, fields) are made by their `@id` as required by the Croissant specification.
- The dataset contains valuable mass spectrometry protein data for extracellular vesicles from human mast cells, suitable for quantitative and categorical analysis.
- See the schema for further details on field meanings and types.

You can adapt this workflow for other Croissant-formatted datasets by updating the URL and referencing the entities via their `@id`s.